# Movie Reviews ML Project

# Part I: Bag of Words

In [1]:
#adapted from:
#https://github.com/sahilee26/IMDB-Movie-Reviews-Sentiment-Analysis/blob/master/Bag-of-words-random-forest.ipynb
#https://github.com/shiaoligreen/practical-data-science/tree/master/Bag%20of%20Words%20Meets%20Bags%20of%20Popcorn
#https://github.com/amueller/word_cloud

# Load packages
import pandas as pd  
from bs4 import BeautifulSoup  
import re
import nltk
from nltk.corpus import stopwords as nltkstopwords
import nltk.data
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import logging
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS
import graphviz
from sklearn.tree import export_graphviz

<class 'ModuleNotFoundError'>: No module named 'gensim'

In [2]:
# Load data
train = pd.read_csv("data/labeledTrainData.tsv", 
                    header=0, delimiter="\t", 
                    quoting=3)
# Outputs a pandas dataframe

test = pd.read_csv("data/testData.tsv", 
                   header=0, delimiter="\t",
                   quoting=3 )

unlabeled_train = pd.read_csv("data/unlabeledTrainData.tsv", header=0, 
                              delimiter="\t", quoting=3 )

# Split data to test preprocessing and modeling techniques
X_train, X_test, y_train, y_test = train_test_split(train.drop(columns=['sentiment']), 
                                                    train.sentiment, test_size=0.2)
# One portion to train, Another to evaluate the model
# X is model/input value, Y is the answers --- Given X predict Y

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

<class 'FileNotFoundError'>: [Errno 44] No such file or directory: 'data/labeledTrainData.tsv'

In [ ]:
#clean training reviews with review_to_word()

print(X_train.loc[0])
print(X_train.loc[0]["review"])

cleaned_training_reviews = []
for i in range (len(X_train)):
    cleaned_training_reviews.append(review_to_words(X_train.loc[i]["review"]))

print(cleaned_training_reviews[0])

cleaned_test_reviews = []
for j in range (len(X_test)):
    cleaned_test_reviews.append(review_to_words(X_test.loc[j]["review"]))

print(cleaned_test_reviews[0])

In [ ]:
#print(type(X_train)) # <class 'pandas.core.frame.DataFrame'>
#print(type(y_train)) # <class 'pandas.core.series.Series'>
#print(X_train) # Table with the index, id and review
#print("") # Prints nothing; What did you expect?
#print(X_train.loc[0]) # Prints the 1st element from the table in dictionary-esque format
#print("") # Prints nothing; What did you expect?
print(y_train.loc[::-1]) # Prints 72-75 in 1-train vector form? Length of output directly correlates to 
#print(y_train.loc[4]) # Prints 1 (an answer in vector form?)
#print(X_train.loc[4]) # Prints the element of the table with index 4
# .iloc finds elements based on index while .loc finds based on the original row number ; reset_index resets the index

In [ ]:
# get all the positive review rows from the pandas dataframe
# get every row except when the sentiment for the row is 0
# make a new dataframe from selected rows
# Splits Positive and Negative Reviews
y_train_list = y_train.tolist()
print(type(X_train))
print(X_train)

pos_reviews = []
neg_reviews = []
for idx in range(len(X_train)):
    if y_train_list[idx] == 1:
        pos_reviews.append(X_train.loc[idx]["review"])
    else:
        neg_reviews.append(X_train.loc[idx]["review"])

#print(y_train_list)
#X_train_positive = X_train[i]
#print(X_train_positive)

# get all the negative review rows from the pandas dataframe

print(pos_reviews[0])
print(neg_reviews[0])

In [ ]:
# Download text datasets, including stop words
nltk.download('stopwords')

In [3]:
from nltk.corpus import stopwords as nltkstopwords
print(nltkstopwords.words('french'))
print(nltkstopwords.words('german'))
print(nltkstopwords.words('english'))
print(nltkstopwords.words.__dir__)

<class 'LookupError'>: 
**********************************************************************
  Resource [93mstopwords[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('stopwords')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/stopwords[0m

  Searched in:
    - '/home/pyodide/nltk_data'
    - '/nltk_data'
    - '/share/nltk_data'
    - '/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [ ]:
def review_to_words(review, string = True, remove_stopwords=True):
    '''
    Function to convert a document to a sequence of words, 
    optionally removing stop words.  
    Returns a list or string of words.
    '''
    # Remove HTML
    review_text = BeautifulSoup(review).get_text() # Beautiful Soup is used to clean up text
    # Remove non-letters
    review_text = re.sub("[^a-zA-Z]"," ", review_text)
    # Convert words to lower case and split them
    words = review_text.lower().split()
    # Optionally remove stop words (false by default)
    if remove_stopwords:
        stops = set(nltkstopwords.words("english"))
        words = [w for w in words if not w in stops]
    if string:
        return " ".join(words)
    else:
        return words

In [ ]:
# Can you select a single review (any random review) from the training data and pass it to review_to_words()?  What is the output?

# your code here
# get a single review from X_train
# print the review
# pass it as an argument to review_to_words()
# print the output

print(review_to_words(neg_reviews[0], string=True, remove_stopwords=True))

In [ ]:
sentence = ["I", "did", "not", "like", "the", "movie"]

In [ ]:
def sentence_to_bigrams(sentence: [str]):
    modified_sentence = BeautifulSoup(sentence).get_text()
    modified_sentence = re.sub("[^a-zA-Z]"," ", modified_sentence)
    words = modified_sentence.lower().split()
        
    bigrams = []
    for idx in range(len(words)-1):
        bigrams.append(words[idx:idx+2])

    return bigrams

In [ ]:
print(sentence_to_bigrams(" ".join(sentence)))

In [ ]:
#Can you modify the function above to add a special START and END token in the function above and then get all bigrams?
#Call it "sentence_to_padded_bigrams"
# What is different about the new output?
# What can it capture that the unpadded version cannot?

In [ ]:
def sentence_to_padded_bigrams(sentence: [str]):
    modified_sentence = BeautifulSoup(sentence).get_text()
    modified_sentence = re.sub("[^a-zA-Z]"," ", modified_sentence)
    words = ("<START> " + modified_sentence.lower() + " <END>").split()
        
    bigrams = []
    for idx in range(len(words)-1):
        bigrams.append(words[idx:idx+2])

    return bigrams

In [ ]:
print(sentence_to_padded_bigrams(" ".join(sentence)))

In [ ]:
def review_to_bigrams(review: [str], string = True, remove_stopwords=False):
    '''
    Function to convert a document to a sequence of word bigrams,
    optionally removing stop words.
    Returns a list of bigrams.
    '''
    #E.g., ["I", "liked", "this" ,"movie"] -> ["I liked", "liked this", "this movie"]
    #your code here
    #     
    # Remove HTML
    review_text = BeautifulSoup(review).get_text()
    # Remove non-letters
    review_text = re.sub("[^a-zA-Z]"," ", review_text)
    # Convert words to lower case and split them
    words = review_text.lower().split()
    
    # Optionally remove stop words (false by default)
    if remove_stopwords:
        stops = set(nltkstopwords.words("english"))
        words = [w for w in words if not w in stops]
        
    bigrams = []
    for i in range(len(words)-1):
        #print(i)
        bigrams.append(words[i] + " " + words[i+1])

    return bigrams

In [ ]:
#print(review_to_bigrams(["I", "liked", "this", "movie"]))
my_bigrams =review_to_bigrams("I liked this movie")
print(my_bigrams)

In [ ]:
# Get list of reviews
# hint: use a list comprehension to covert all reviews to words, using the X_train object's 'review'
#clean_train_reviews = 
#clean_test_reviews = 

#print(clean_train_reviews[0])

In [ ]:
# Get lists of reviews using the bigram function instead of the review_to_words function
# your code here
#clean_test_reviews_bigram = [ ? ]
#clean_train_reviews_bigram = [ ? ]
#print(clean_train_reviews_bigram[0])

In [ ]:
# Get words from positive training reviews (liked the movie) only
# Get words from negative training reviews (disliked the movie) only
# your code here
def detect_sentiment(reviews_list: [list]):
    wordcloud = {}
    for idx in range (len(reviews_list)):
        words = review_to_words(reviews_list[idx], string=False, remove_stopwords=True)
        for idx in range (len(words)):
            if words[idx] in wordcloud:
                wordcloud[words[idx]]+= 1
            else:
                wordcloud[words[idx]] = 1
    sorted_wordcloud = dict(sorted(wordcloud.items(), key=lambda item: item[1], reverse = True))
    return sorted_wordcloud
                
pos_wordcloud = detect_sentiment(pos_reviews)
neg_wordcloud = detect_sentiment(neg_reviews)
# positive_train_reviews = [review_to_words(X_train[ ? ][i]) for i in range(len(X_train))]
# positive_train_reviews_bigram = [ ? for i in range(len(X_train))]

In [ ]:
#print(pos_reviews[0:5])

cleaned_pos_reviews = [ review_to_words(review) for review in pos_reviews ]
cleaned_neg_reviews = [ review_to_words(review) for review in neg_reviews ]
print(cleaned_pos_reviews[0])

In [ ]:
# What do you think are some common words in positive movie reviews?  Write out your top 10.
# 1. captivating
# 2. masterpiece
# 3. wonderful 
# 4. generational
# 5. brilliant
# 6. whimsical
# 7. amazing
# 8.
# 9.
# 10.

In [ ]:
# What about negative words?
# 1. disgusting
# 2. horrible
# 3. janky
# 4. confusing
# 5. dreadful
# 6. plot
# 7. terrible
# 8.
# 9.
# 10.

In [ ]:
# Initialize a bag of words  
vectorizer = CountVectorizer(analyzer = "word",   
                             tokenizer = None,    
                             preprocessor = None, 
                             stop_words = None,   
                             max_features = 5000) 

# Fit transform the data
train_feat = vectorizer.fit_transform(cleaned_training_reviews).toarray()
test_feat = vectorizer.transform(cleaned_test_reviews).toarray()

In [ ]:
print(type(train_feat))
print(train_feat[0])

In [ ]:
# Let's visualize the words using a wordcloud
stopwords = set(STOPWORDS)

def show_wordcloud(data: [dict], title = None):
    wordcloud = WordCloud(
        background_color='black',
        stopwords=stopwords,
        max_words=200,
        max_font_size=40, 
        scale=3,
        random_state=1 # chosen at random by flipping a coin; it was heads
).generate_from_frequencies(data)
    # 3. Your existing plotting code remains the same:
    fig = plt.figure(1, figsize=(15, 15))
    plt.axis('off')
    if title: 
        fig.suptitle(title, fontsize=20)
        fig.subplots_adjust(top=2.3)
    
    plt.imshow(wordcloud)
    plt.show()

#print("Positive")
#show_wordcloud(pos_wordcloud)
#print("Negative")
#show_wordcloud(neg_wordcloud)

common_pos_words = list(pos_wordcloud.keys())[:100]
common_neg_words = list(neg_wordcloud.keys())[:100]
common_neutral_words = [word for word in common_pos_words if word in common_neg_words]
filtered_pos_wordcloud = {k: v for k, v in pos_wordcloud.items() if k not in common_neutral_words}
filtered_neg_wordcloud = {k: v for k, v in neg_wordcloud.items() if k not in common_neutral_words}
#print(common_neutral_words)

print("Filtered Positive")
show_wordcloud(filtered_pos_wordcloud)
print(list(filtered_pos_wordcloud.keys())[:100])
print("Filtered Negative")
show_wordcloud(filtered_neg_wordcloud)
print(list(filtered_neg_wordcloud.keys())[:100])

In [ ]:
#print(clean_train_reviews[0])
#print(clean_train_bigram_reviews[0])

def flatten_bigram(bigram_list):
    big_string = ""
    #iterate over bigrams
    for bigram in bigram_list:
        word1, word2 = bigram.split() #convert bigram from string with spaces to combined, spaceless string
        big_string = big_string + word1 + word2 + " "#combine all spaceless strings  into one string
    return big_string
combined_reviews = [flatten_bigram(x) for x in clean_train_bigram_reviews]
print(combined_reviews[0])

show_wordcloud(combined_reviews[:10])

In [ ]:
# Can you make separate word clouds for positive vs. negative reviews?  Do you see any of your top 10 in the word clouds?
cleaned_pos_reviews = [ review_to_words(review) for review in pos_reviews ]
cleaned_neg_reviews = [ review_to_words(review) for review in neg_reviews ]
print(cleaned_pos_reviews[0])

show_wordcloud(cleaned_pos_reviews)
# your code here

In [ ]:
show_wordcloud(cleaned_neg_reviews)

In [ ]:
# Take a look at the vocabulary
vocab = vectorizer.get_feature_names_out()
print(vocab)

In [ ]:
# Random Forest Classifier Overview
#
# Improved performance variant of Decision Tree Learning, a supervised learning algorithm
#
# https://en.wikipedia.org/wiki/Decision_tree_learning#/media/File:Decision_Tree.jpg
#
# Decision trees are ways of taking input, and then moving left or right depending on your input and coming to a final decision, kind of like turn-by-turn instructions.
#
# For example, passenger survival on the Titanic can be predicted (not perfectly!) using features like sex and age.  Deciding which features and values to put where in the tree comes from taking measurements on the training data.
#
# Some features and values are better at splitting the output into the target predictions than others.  For example, wearing brown clothes vs. wearing black clothes is probably not very predictive of whether passengers survived or not, whereas age appears to be more predictive.
# 
# Let's imagine that one of the nodes in the tree is clothing color.  If you put all the data through the node, the two children nodes resulting from black or brown clothes would likely have something like 50% died and 50% survived, that is, clothing color doesn't separate the people into the target categories very well.
#
# On the other hand, for a node like 'age', being a child or adult could have 90/10 split for children surviving and 20/80 split for adults surviving.  This is a better decision to make when predicting survival because there is higher 'purity' in the outcomes - they're more skewed toward a category of interest than a simple 50/50 split.
#
# This ability to differentiate, or selectivity, is called 'information gain' and can be measured mathematically a few ways.

In [ ]:
# Get predicitons
def get_preds(test_feat, train_feat, y_test, y_train, model, title='Random Forest'):
    model.fit(train_feat, y_train)
    preds = model.predict(test_feat)
    
    #accuracy = accuracy_score(y_test, y_pred)
    #print("Accuracy:", accuracy)

    fpr, tpr, _ = roc_curve(y_test, preds)
    roc_auc = auc(fpr, tpr)
    print('AUC:', roc_auc)
    
    #F1 doesn't matter because of class balance here
    
    # plot AUC
    plt.plot(fpr, tpr)
    plt.title(title)
    plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.show()
    
    return preds, model

In [ ]:
# Area Under the Curve Definition - https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc
#
# How do you measure model performance?  We're dealing with a binary classification (good review vs. bad review), and we want to know how often our model made the right prediction.
#
# True and false positive rates are one way of measuring correctness.
#
# A true positive is when the model predicts category A and the correct label is category A.
# A false positive is when the model predicts category A the correct label is some other category (not category A).

# A true negative is when the model correctly predicts some other category (!A). 
# A false negative is when the model incorrectly predicts some other category (!A).  The true label is A, but the model predicted something else.

# True positive rate (i.e., recall) is true positives / (true positives + false negatives) 
# False positive rate is false positives / (false positives + true negatives)
#
# Better models have a low false positive rate and a high true positive rate.  If they got all true positives and no false positives, they would be at (0,1) in the AUC graph, and the space between the point and the diagonal line would be maximised.  More area means higher performance.
#
# There are other ways of measuring performance.  

In [ ]:
len(vocab)
#with open('/tmp/fst.svg', 'w') as f:
#    f.write(t._repr_svg_())

In [ ]:
preds_rf, model = get_preds(test_feat, train_feat, y_test, y_train,
                  RandomForestClassifier(n_estimators = 100))

In [ ]:
 # visualize decision tree from classifier
for i in range(10):
    tree = model.estimators_[i]
    dot_data = export_graphviz(tree,
                            feature_names=vocab,  
                            filled=True,  
                           max_depth=2, 
                           impurity=False, 
                           proportion=True)
    graph = graphviz.Source(dot_data)
    display(graph)

In [ ]:
def get_accuracy(preds, labels):
    correct = 0
    assert len(preds) == len(labels)
    for i in range(len(preds)):
        if preds[i] == labels[i]:
            correct += 1
    print("Accuracy: {}".format(correct/len(preds)))

preds = preds_rf.tolist()
labels = y_test.tolist()

get_accuracy(preds, labels)

In [ ]:
print(preds_rf[0]) #first prediction for test item - model predicted positive sentiment
print(test_feat[0]) #first test review
print(clean_test_reviews[0])
print(type(y_test))
print(y_test.iloc[0])

In [ ]:
preds_nb = get_preds(test_feat, train_feat, 
                  y_test, y_train, 
                  MultinomialNB(), title='Naive Bayes')

In [ ]:
# Are bigrams better or worse features compared to Bag of Words? Make a prediction.
# https://www.pythonprogramming.in/use-sklearn-countvectorize-vocabulary-specification-with-bigrams.html

# your code here

# Is the performance better or worse?

# Why do you think that is?

In [ ]:
# Naive Bayes assumes independence of features, which is not always the case (rarely the case with language)

# Try other model parameters (e.g., change num_estimators for Random Forest model).  Do you think more estimators will improve or harm model performance?

# Find some other models in sci-kit learn.  Are any better than Naive Bayes?



In [ ]:
#Try calculating some other evaluation metrics

# your code here

# Calculate accuracy

# Calculate precision

# Calculate recall

# Calculate F1

In [ ]:
movie_review = "I love Titanic because 1. Leonardo DiCaprio is fine ash 2. it has a ship blowing up!!!"

cleaned_review = review_to_words(movie_review)
review_feat = vectorizer.transform([cleaned_review]).toarray()

print(len(review_feat[0]))
print(type(review_feat))


In [ ]:
movie_neg_review = "I hate Titanic because Rose Dewitt Bukater is a stupid person with a stupid name"

cleaned_neg_review = review_to_words(movie_neg_review)
review_neg_feat = vectorizer.transform([cleaned_neg_review]).toarray()

print(len(review_neg_feat[0]))
print(type(review_neg_feat))

In [ ]:
prediction = model.predict(review_feat)
print(prediction)
neg_prediction = model.predict(review_neg_feat)
print(neg_prediction)

# Part II: Word2Vec

In [ ]:
# Download the punkt tokenizer if necessary
nltk.download('punkt', download_dir='tokenizers')   

# Load the punkt tokenizer
tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')

# Define a function to split a review into parsed sentences
def review_to_sentences( review, tokenizer, remove_stopwords=False ):
    '''
    Split a review into parsed sentences. 
    Returns a list of sentences, where each sentence is a list of words
    '''
    # Remove HTML
    review = BeautifulSoup(review).get_text()
    # Use the NLTK tokenizer to split the paragraph into sentences
    raw_sentences = tokenizer.tokenize(review.strip())
    #
    # Loop over each sentence
    sentences = []
    for raw_sentence in raw_sentences:
        # If a sentence is empty, skip it
        if len(raw_sentence) > 0:
            # Get a list of words
            sentences.append(review_to_words(raw_sentence, string=False, 
                                             remove_stopwords=False))
    return sentences

In [ ]:
sentences = []

for review in X_train["review"]:
    sentences += review_to_sentences(review, tokenizer)

for review in unlabeled_train["review"]:
    sentences += review_to_sentences(review, tokenizer)

In [ ]:
# Set parameters
num_features = 300                   
min_word_count = 40                         
num_workers = 4       
context = 10                                                                                      
downsampling = 1e-3

# Initialize and train the model
model = Word2Vec(sentences, workers=num_workers,
                 vector_size=num_features, min_count = min_word_count,
                 window = context, sample = downsampling)

# If you don't plan to train the model any further, calling 
# init_sims will make the model much more memory-efficient.
#model.init_sims(replace=True)
#4mins?

In [ ]:
def makeFeatureVec(words, model, num_features):
    '''Average all of the word vectors in a paragraph'''
    # Pre-initialize an empty numpy array (for speed)
    featureVec = np.zeros((num_features,),dtype="float32")
    nwords = 0
    # ICreate a set of words on voc 
    index2word_set = set(model.wv.index2word)
    # Add feature vectors if they are inside model vocabulary
    for word in words:
        if word in index2word_set: 
            nwords = nwords + 1
            featureVec = np.add(featureVec,model[word])
    # Get averages
    featureVec = np.divide(featureVec,nwords)
    return featureVec


def getAvgFeatureVecs(reviews, model, num_features):
    '''
    Given a set of reviews (each one a list of words), calculate 
    the average feature vector for each one and return a 2D numpy array 
    '''
    # Initialize a counter
    counter = 0
    # Preallocate a 2D numpy array, for speed
    reviewFeatureVecs = np.zeros((len(reviews),num_features),dtype="float32")
    # Loop through the reviews
    for review in reviews:
        reviewFeatureVecs[counter] = makeFeatureVec(review, model,
                                                   num_features)
        counter = counter + 1
    return reviewFeatureVecs

In [ ]:
# Calculate average feature vectors for training and testing sets,

clean_train_reviews = [review_to_words(review, string=False, remove_stopwords=True) for review in X_train.review]

trainDataVecs = getAvgFeatureVecs(clean_train_reviews, model, num_features)

clean_test_reviews = [review_to_words(review, string=False, remove_stopwords=True) for review in X_test.review]

testDataVecs = getAvgFeatureVecs(clean_test_reviews, model, num_features)

In [ ]:
# Get Results
preds_rf1 = get_preds(testDataVecs, trainDataVecs, 
                      y_test, y_train, 
                      RandomForestClassifier(n_estimators = 100))

In [ ]:
# Use Gaussian NB because Multinomial doesn't work with negative feature values
preds_nb1 = get_preds(testDataVecs, trainDataVecs, 
                      y_test, y_train, 
                      GaussianNB(), title='Naive Bayes')

## Tf-Idf

This is short for term frequency-inverse document frequency. 

Term frequency (the same as bag of words):  $$\text{Tf} = \frac{\text{number of times term appears in document}}{\text{total number of words in document}}$$

Inverse document frequency measures how rare a term is across all documents (the higher the value, the rarer the word): $$\text{Idf} = \ln\left(\frac{\text{total number of documents}}{\text{number of documents with term in them}}\right)$$

We combine these two to get Tf-Idf as follows:
 $$\text{Tf-idf} = \text{Tf} \times \text{Idf}$$

 https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
 

In [ ]:
# TFIDF train set
tfidf_transformer = TfidfTransformer().fit(train_feat)
train_tfidf = tfidf_transformer.transform(train_feat)

# apply tfidf to test set
test_tfidf = tfidf_transformer.transform(test_feat)

In [ ]:
# Get Results
preds_rf2 = get_preds(test_tfidf, train_tfidf, 
                  y_test, y_train, 
                  RandomForestClassifier(n_estimators = 100))

In [ ]:
preds_nb2 = get_preds(test_tfidf, train_tfidf, 
                  y_test, y_train, 
                  MultinomialNB(), title='Naive Bayes')